# KnottedGraph: Yamada engine benchmark

Paper-facing benchmark for `Latest_Workplace`. It compares four routes: (1) production direct Yamada recursion with memoization, (2) the same recurrence/shortcuts with memoization removed, (3) production `method="negami"` (`NegamiRecursiveEvaluator`), and (4) the explicit Negami edge-subset sum retained as an independent small-graph reference.

Metrics: median wall time, peak traced Python memory, recursive calls, memo entries/cache-hit rate, graph size/irregularity, and scaling with planar crossing number $c$ and $3^c$ resolved states. Run `MODE="quick"` first; use `MODE="paper"` only for manuscript data.

**Important:** the public Negami backend is recursive; do not describe it as the explicit edge-subset sum in the paper.

In [ ]:
from pathlib import Path
from contextlib import contextmanager
import csv,gc,importlib.util,json,os,platform,signal,statistics,subprocess,sys,threading,time,tracemalloc
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import sympy as sp

def root():
    p=Path.cwd().resolve()
    for q in (p,*p.parents):
        if (q/'pyproject.toml').exists() and (q/'src/knotted_graph').exists(): return q
    raise RuntimeError('Open this notebook from inside the KnottedGraph repository.')
ROOT=root(); sys.path.insert(0,str(ROOT/'src'))
OUT=ROOT/'User_guide/benchmarks'; RES=OUT/'results'; FIG=OUT/'figures'; RES.mkdir(parents=True,exist_ok=True); FIG.mkdir(parents=True,exist_ok=True)
import knotted_graph.invariants.yamada.recursive as yr
import knotted_graph.invariants.yamada.polynomial as yp
from knotted_graph.invariants.yamada.polynomial import Yamada
from knotted_graph.projection import select_projection
A=sp.Symbol('A'); MODE='quick'
REPEATS,TIMEOUT,MAX_SUBSET_E,MAX_NOMEMO_E,CROSSINGS=(2,15,12,16,range(6)) if MODE=='quick' else (5,120,16,22,range(8))
def git(*a):
    try:return subprocess.check_output(['git',*a],cwd=ROOT,text=True).strip()
    except:return 'unavailable'
prov=dict(mode=MODE,branch=git('rev-parse','--abbrev-ref','HEAD'),commit=git('rev-parse','HEAD'),python=sys.version,platform=platform.platform(),cpu=os.cpu_count(),numpy=np.__version__,networkx=nx.__version__,sympy=sp.__version__)
print(json.dumps(prov,indent=2)); (RES/'benchmark_provenance.json').write_text(json.dumps(prov,indent=2))

## Measurement protocol

Each repetition starts with a fresh evaluator and a cleared SymPy cache. `tracemalloc` gives comparative peak Python allocations; if the final paper needs strict OS peak RSS, add a subprocess/RSS benchmark later. A Unix timer prevents pathological no-memo/subset-sum cases from hanging the notebook.

In [ ]:
class Timeout(RuntimeError):pass
@contextmanager
def limit(sec):
    ok=hasattr(signal,'setitimer') and threading.current_thread() is threading.main_thread()
    if not ok: yield; return
    old=signal.getsignal(signal.SIGALRM); signal.signal(signal.SIGALRM,lambda *x:(_ for _ in ()).throw(Timeout())) ; signal.setitimer(signal.ITIMER_REAL,sec)
    try:yield
    finally:signal.setitimer(signal.ITIMER_REAL,0); signal.signal(signal.SIGALRM,old)
def measure(fn):
    gc.collect(); sp.core.cache.clear_cache(); tracemalloc.start(); t=time.perf_counter(); status='ok'; err=''; val=None
    try:
        with limit(TIMEOUT):val=fn()
    except Timeout:status,err='timeout',f'>{TIMEOUT}s'
    except Exception as e:status,err='error',f'{type(e).__name__}: {e}'
    dt=time.perf_counter()-t; _,peak=tracemalloc.get_traced_memory(); tracemalloc.stop(); return dict(status=status,error=err,elapsed_s=dt,peak_mb=peak/2**20,value=val)
class PY(yr.YamadaRecursiveEvaluator):
    def __init__(self,A):super().__init__(A);self.calls=self.q=self.h=0
    def _rec(self,G):self.calls+=1;return super()._rec(G)
    def _cache_get(self,k):self.q+=1;v=super()._cache_get(k);self.h+=v is not None;return v
class PN(yr.NegamiRecursiveEvaluator):
    def __init__(self,x,y):super().__init__(x,y);self.calls=self.q=self.h=0
    def _rec(self,G):self.calls+=1;return super()._rec(G)
    def _cache_get(self,k):self.q+=1;v=super()._cache_get(k);self.h+=v is not None;return v
class NoMemo:
    def __init__(self,A):self.s=A+1+A**-1;self.calls=0
    def F(self,x):return sp.simplify(x)
    def compute(self,G):return self.F(self.r(G))
    def r(self,H):
        self.calls+=1;H=yr.normalize_multigraph(H);n=H.number_of_nodes()
        if H.number_of_edges()==0:return self.F((-1)**n)
        cc=yr.connected_components_ignoring_loops(H)
        if len(cc)>1:return self.F(sp.prod(self.r(H.subgraph(c).copy()) for c in cc))
        if yr.has_isthmus_multigraph(H):return sp.Integer(0)
        if yr.is_cycle_multigraph(H):return self.s
        s=yr.theta_edge_count(H)
        if s is not None:return self.F(sum((-1)**(p-1)*self.s**p for p in range(1,s)))
        e=yr._pick_loop_edge(H)
        if e is not None:return self.F(-self.s*self.r(yr.delete_multigraph_edge(H,e)))
        parts=yr._split_at_articulation(H)
        if parts is not None:return self.F((-1)**(len(parts)-1)*sp.prod(self.r(p) for p in parts))
        e=yr.pick_nonloop_edge(H);return self.F(self.r(yr.delete_multigraph_edge(H,e))+self.r(yr.contract_multigraph_edge(H,e)))
def evaluate(G,b):
    if b=='recursive_memo':
        e=PY(A);z=e.compute(G);s=dict(calls=e.calls,cache_entries=len(e.memo),cache_hit_rate=e.h/e.q if e.q else 0)
    elif b=='recursive_no_memo':e=NoMemo(A);z=e.compute(G);s=dict(calls=e.calls,cache_entries=0,cache_hit_rate=0)
    elif b=='negami_recursive':
        x,y=sp.symbols('x y');e=PN(x,y);z=e.compute(G).xreplace({x:-1,y:-A-2-A**-1});s=dict(calls=e.calls,cache_entries=len(e.memo),cache_hit_rate=e.h/e.q if e.q else 0)
    else:
        x,y=sp.symbols('x y');z=yp.compute_negami(G,x,y).xreplace({x:-1,y:-A-2-A**-1});s=dict(subset_states=2**G.number_of_edges())
    return sp.expand(sp.cancel(z)),s

## Graph suite

Use bridge-free families so the isthmus shortcut does not trivialize timing: wheels (structured/degree-irregular), circular ladders (structured/regular), random bridgeless 3-regular graphs, and chorded cycles (controlled irregularity).

In [ ]:
def MG(G):
    H=nx.MultiGraph();H.add_nodes_from(G);H.add_edges_from(G.edges());return H
def chorded(n,k,seed):
    r=np.random.default_rng(seed);G=nx.cycle_graph(n);p=[(i,j) for i in range(n) for j in range(i+1,n) if not G.has_edge(i,j)];r.shuffle(p);G.add_edges_from(p[:k]);return MG(G)
def reg3(n,seed):
    for k in range(100):
        G=nx.random_regular_graph(3,n,seed=seed+k)
        if nx.is_connected(G) and not list(nx.bridges(G)):return MG(G)
def desc(G):
    d=np.array([x for _,x in G.degree()],float);V=G.number_of_nodes();E=G.number_of_edges();return dict(V=V,E=E,cyclomatic=E-V+1,mean_degree=d.mean(),degree_cv=d.std()/d.mean())
CASES=[]
for n in ([5,6,7] if MODE=='quick' else [5,6,7,8,9,10]):CASES.append((f'wheel{n}','wheel',MG(nx.wheel_graph(n))))
for n in ([3,4,5] if MODE=='quick' else [3,4,5,6,7]):CASES.append((f'ladder{n}','ladder',MG(nx.circular_ladder_graph(n))))
for n in ([6,8] if MODE=='quick' else [6,8,10,12,14]):CASES.append((f'random3_{n}','random3',reg3(n,100+n)))
for n,k in ([(7,2),(8,3),(9,4)] if MODE=='quick' else [(8,2),(8,4),(10,3),(10,5),(12,4),(12,6)]):CASES.append((f'chorded{n}_{k}','chorded',chorded(n,k,1000+n+k)))
print([(x,f,desc(g)) for x,f,g in CASES])
# correctness gate
s=A+1+A**-1;assert sp.simplify(evaluate(MG(nx.cycle_graph(5)),'recursive_memo')[0]-s)==0
G0=chorded(7,2,123);ref=evaluate(G0,'recursive_memo')[0]
for b in ['recursive_no_memo','negami_recursive','negami_subset_sum']:assert sp.simplify(sp.cancel(evaluate(G0,b)[0]-ref))==0,b
print('Correctness gate passed.')

## Main internal benchmark

The timed expression excludes the symbolic equality check. Expensive methods are automatically skipped above chosen edge thresholds. CSV output is the source of truth for manuscript tables/figures.

In [ ]:
B=['recursive_memo','recursive_no_memo','negami_recursive','negami_subset_sum'];rows=[]
def med(a,k):
    v=[x[k] for x in a if x['status']=='ok'];return statistics.median(v) if v else float('nan')
for cid,fam,G in CASES:
    D=desc(G);case=[];print('\n',cid,D)
    for b in B:
        if b=='recursive_no_memo' and D['E']>MAX_NOMEMO_E:r=dict(status='skipped',error='edge threshold')
        elif b=='negami_subset_sum' and D['E']>MAX_SUBSET_E:r=dict(status='skipped',error='edge threshold')
        else:
            rr=[];vals=[];ss=[]
            for _ in range(REPEATS):
                m=measure(lambda b=b:evaluate(G,b))
                if m['status']=='ok':v,s=m.pop('value');vals.append(v);ss.append(s)
                else:m.pop('value',None)
                rr.append(m)
                if m['status']!='ok':break
            st='ok' if all(x['status']=='ok' for x in rr) else rr[-1]['status'];r=dict(status=st,error='' if st=='ok' else rr[-1]['error'],elapsed_s=med(rr,'elapsed_s'),peak_mb=med(rr,'peak_mb'))
            if ss:
                for k in ['calls','cache_entries','cache_hit_rate','subset_states']:
                    v=[s[k] for s in ss if k in s];r[k]=statistics.median(v) if v else float('nan')
                r['_expr']=vals[0]
        z=dict(mode=MODE,case_id=cid,family=fam,backend=b,**D,**r);rows.append(z);case.append(z);print(b,z['status'],z.get('elapsed_s'))
    base=next((x for x in case if x['backend']=='recursive_memo' and x['status']=='ok'),None)
    if base:
        for x in case:
            if x['status']=='ok' and '_expr' in x:x['agrees']=sp.simplify(sp.cancel(x['_expr']-base['_expr']))==0
def savecsv(path,data):
    c=[{k:v for k,v in r.items() if not k.startswith('_')} for r in data];keys=list(dict.fromkeys(k for r in c for k in r));f=open(path,'w',newline='');w=csv.DictWriter(f,fieldnames=keys);w.writeheader();w.writerows(c);f.close()
savecsv(RES/f'yamada_internal_{MODE}.csv',rows)

## Core figures

The production-backend ratio is $R=t_{recursive}/t_{negami}$. Therefore $R>1$ means Negami is faster and $R<1$ means direct recursion is faster. Only call this a crossover if the data contain both regimes.

In [ ]:
def save(n):plt.savefig(FIG/f'{n}_{MODE}.png',dpi=300,bbox_inches='tight');plt.savefig(FIG/f'{n}_{MODE}.pdf',bbox_inches='tight')
def ok(b):return [r for r in rows if r['backend']==b and r['status']=='ok']
plt.figure(figsize=(7,5))
for b in B:
 q=sorted(ok(b),key=lambda r:r['E']);plt.plot([r['E'] for r in q],[r['elapsed_s'] for r in q],'o-',label=b) if q else None
plt.yscale('log');plt.xlabel('Edges E');plt.ylabel('Median runtime (s)');plt.legend();plt.grid(alpha=.25);save('runtime_vs_edges');plt.show()
idx={(r['case_id'],r['backend']):r for r in rows if r['status']=='ok'};speed=[];cross=[]
for cid,f,G in CASES:
 m=idx.get((cid,'recursive_memo'));n=idx.get((cid,'recursive_no_memo'));g=idx.get((cid,'negami_recursive'))
 if m and n:speed.append((cid,f,m['E'],n['elapsed_s']/m['elapsed_s']))
 if m and g:cross.append((cid,f,m['E'],m['elapsed_s']/g['elapsed_s']))
for data,name,ylabel in [(speed,'memoization_speedup','no-memo / memo runtime'),(cross,'backend_crossover','recursive / negami runtime')]:
 plt.figure(figsize=(7,5))
 for f in sorted(set(x[1] for x in data)):
  q=[x for x in data if x[1]==f];plt.scatter([x[2] for x in q],[x[3] for x in q],label=f)
 plt.axhline(1);plt.yscale('log');plt.xlabel('Edges E');plt.ylabel(ylabel);plt.legend();plt.grid(alpha=.25);save(name);plt.show()
plt.figure(figsize=(7,5))
for b in ['recursive_memo','negami_recursive']:
 q=sorted(ok(b),key=lambda r:r['E']);plt.plot([r['E'] for r in q],[r['peak_mb'] for r in q],'o-',label=b)
plt.xlabel('Edges E');plt.ylabel('Peak traced Python memory (MB)');plt.legend();plt.grid(alpha=.25);save('memory_vs_edges');plt.show()

## Crossing-number benchmark

Construct a theta embedding with a controlled zig-zag pair of strands, verify the detected crossing count at fixed projection, then time **invariant evaluation separately from projection**. Record $3^c$ states and the number of distinct state topologies for moderate $c$. This directly tests the crossing-resolution bottleneck and state reuse.

In [ ]:
def crossed_theta(c):
 u=np.array([-1.,0.,0.]);v=np.array([c+1.,0.,0.]);x=np.arange(c+1,dtype=float);s=np.where(np.arange(c+1)%2==0,1.,-1.);p1=np.column_stack([x,s,np.full(c+1,.6)]);p2=np.column_stack([x,-s,np.full(c+1,-.6)]);p3=np.array([u,[-.25,2.8,0.],[c+.25,2.8,0.],v]);G=nx.MultiGraph();G.add_node('u',pos=u);G.add_node('v',pos=v);G.add_edge('u','v',pts=np.vstack([u,p1,v]));G.add_edge('u','v',pts=np.vstack([u,p2,v]));G.add_edge('u','v',pts=p3);return G
cr=[]
for requested in CROSSINGS:
 pm=measure(lambda:select_projection(crossed_theta(requested),rotation_angles=(0.,0.,0.)))
 if pm['status']!='ok':print('projection failed',requested,pm['error']);continue
 P=pm['value'];c=P.num_crossings;comp=Yamada.from_PDCode(P.processor);unique=len({yr.multigraph_key(g) for g,e in comp._iter_state_graphs()}) if c<=7 else float('nan');print('requested/detected',requested,c)
 for method in ['recursive','negami']:
  rr=[];vals=[]
  for _ in range(REPEATS):
   m=measure(lambda method=method:comp.compute(A,normalize=False,n_jobs=1,method=method));vals.append(m['value']) if m['status']=='ok' else None;m.pop('value',None);rr.append(m)
   if m['status']!='ok':break
  st='ok' if all(x['status']=='ok' for x in rr) else rr[-1]['status'];cr.append(dict(mode=MODE,requested_crossings=requested,detected_crossings=c,resolved_states=3**c,unique_state_topologies=unique,redundancy=3**c/unique if unique else float('nan'),method=method,status=st,error='' if st=='ok' else rr[-1]['error'],projection_time_s=pm['elapsed_s'],elapsed_s=med(rr,'elapsed_s'),peak_mb=med(rr,'peak_mb'),_expr=vals[0] if vals else None))
 q=[r for r in cr if r['detected_crossings']==c and r['status']=='ok'];
 if len(q)==2:q[0]['agrees']=q[1]['agrees']=sp.simplify(sp.cancel(q[0]['_expr']-q[1]['_expr']))==0
savecsv(RES/f'yamada_crossings_{MODE}.csv',cr)
plt.figure(figsize=(7,5))
for m in ['recursive','negami']:
 q=sorted([r for r in cr if r['method']==m and r['status']=='ok'],key=lambda r:r['detected_crossings']);plt.plot([r['detected_crossings'] for r in q],[r['elapsed_s'] for r in q],'o-',label=m)
plt.yscale('log');plt.xlabel('Crossings c');plt.ylabel('Invariant runtime (s)');plt.legend();plt.grid(alpha=.25);save('runtime_vs_crossings');plt.show()
q=sorted({r['detected_crossings']:r for r in cr if r['status']=='ok'}.values(),key=lambda r:r['detected_crossings']);plt.figure(figsize=(7,5));plt.plot([r['detected_crossings'] for r in q],[r['resolved_states'] for r in q],'o-',label='$3^c$');u=[r for r in q if np.isfinite(r['unique_state_topologies'])];plt.plot([r['detected_crossings'] for r in u],[r['unique_state_topologies'] for r in u],'s-',label='distinct state topologies');plt.yscale('log');plt.xlabel('Crossings c');plt.ylabel('State count');plt.legend();plt.grid(alpha=.25);save('state_growth');plt.show()

## Evidence summary and next benchmark

Do not pre-decide the conclusion. Report memoization gains only if measured; claim a production-backend crossover only if each backend wins on at least one case. The generated CSV files remain the source data.

External packages (`UIUC-ESDL/Yamada`, Topoly) should be benchmarked in a **second, representation-matched notebook** only after verifying that both libraries receive the same mathematical diagram/graph; otherwise conversion/projection policy contaminates wall-time comparisons.

In [ ]:
mb=[x for x in speed if x[3]>1];nf=[x for x in cross if x[3]>1];rf=[x for x in cross if x[3]<1];ag=[r for r in rows if r['status']=='ok' and 'agrees' in r]
print(f'Memoized recursion faster in {len(mb)}/{len(speed)} comparable cases')
if mb:print('Largest memoization speedup:',max(x[3] for x in mb))
print('Negami faster:',len(nf),'Recursive faster:',len(rf),'Two-sided crossover:',bool(nf and rf))
print('Internal symbolic agreement:',sum(bool(r['agrees']) for r in ag),'/',len(ag))
print('External packages installed:',{'yamada':importlib.util.find_spec('yamada') is not None,'topoly':importlib.util.find_spec('topoly') is not None})
print("For manuscript data: restart kernel, set MODE='paper', Run All, and retain benchmark_provenance.json.")